# One-Team Codenames (Blue) — GoogleNews Word2Vec Bots

This notebook runs a **single-team** Codenames variant on the fixed boards provided in `difficulty_boards.txt`.

- **Blue team (OUR)**: 8 words (goal is to guess all)
- **Neutral words**: includes both `OPP` and `NEUTRAL` from the file (treated the same)
- **Assassin**: instant loss if guessed
- **Spymaster bot**: generates `(clue, number)` using GoogleNews Word2Vec
- **Operative bot**: guesses using similarity to the clue

At the end, we report **per-board** and **per-level** metrics:
- Avg turns
- Words/turn (blue words found per turn)
- Accuracy (blue words found / 8)
- Assassin rate


In [1]:
# If needed (Colab): install dependencies
!pip -q install gensim numpy pandas tqdm


## 1) Load GoogleNews Word2Vec

Place `GoogleNews-vectors-negative300.bin.gz` (or `.bin`) next to this notebook, or update `W2V_PATH`.

Tip: during development you can set `LIMIT_VECTORS` to something like 200k–500k to reduce memory/time.


In [2]:
import os
import re
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Tuple, Set, Dict

from gensim.models import KeyedVectors

W2V_PATH = "GoogleNews-vectors-negative300.bin.gz"  # <-- update if needed
LIMIT_VECTORS = 1500000  # set None for full model

assert os.path.exists(W2V_PATH), (
    f"Couldn't find {W2V_PATH}. Put the GoogleNews vectors file next to this notebook "
    "or update W2V_PATH to the correct path."
)

print("Loading Word2Vec model...")
w2v = KeyedVectors.load_word2vec_format(W2V_PATH, binary=True, limit=LIMIT_VECTORS)
print("Loaded vectors:", len(w2v))


Loading Word2Vec model...
Loaded vectors: 1500000


## 2) Utilities

In [3]:
WORD_RE = re.compile(r"^[a-z]+$")

def normalize_token(s: str) -> str:
    return s.strip().lower().replace(" ", "_")

def in_vocab(model: KeyedVectors, token: str) -> bool:
    return token in model.key_to_index

def cosine_sim(model: KeyedVectors, a: str, b: str) -> float:
    return float(model.similarity(a, b))

def mean_vector(model: KeyedVectors, tokens: List[str]) -> np.ndarray:
    vecs = [model[t] for t in tokens if in_vocab(model, t)]
    if not vecs:
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean(np.vstack(vecs), axis=0)


BANNED_CLUES = {
    "nr","mr","ms","mrs","dr","st","jr","sr",
    "re","im","ur","ya","ok","pm","am",
    "nov","oct","sep","aug","jul","jun","apr","mar","feb","jan","dec",
}

def is_legal_clue(clue: str, board_words: Set[str], model: KeyedVectors, threshold: float) -> bool:
    if clue in board_words:
        return False
    if "_" in clue:
        return False
    if not WORD_RE.match(clue):
        return False

    # NEW: ban very short clues and common junk abbreviations
    if len(clue) <= 2:
        return False
    if clue in BANNED_CLUES:
        return False

    for w in board_words:
        if in_vocab(model, w) and in_vocab(model, clue):
            if cosine_sim(model, clue, w) >= threshold:
                return False
    return True


## 3) Game setup

In [4]:
@dataclass
class GameConfig:
    n_blue: int = 8
    illegal_clue_similarity_threshold: float = 0.55
    clue_candidate_topn: int = 50000
    clue_number_cap: int = 8
    operative_extra_guess: bool = False  # set True if you want the +1 guess rule

@dataclass
class GameState:
    blue: Set[str]
    assassin: str
    bystanders: Set[str]   # includes OPP + NEUTRAL from the file
    guessed: Set[str]
    used_clues: Set[str] = field(default_factory=set)

    @property
    def board_all(self) -> Set[str]:
        return set(self.blue) | {self.assassin} | set(self.bystanders)

    @property
    def remaining_blue(self) -> Set[str]:
        return set(self.blue) - set(self.guessed)

    @property
    def remaining_unguessed(self) -> Set[str]:
        return set(self.board_all) - set(self.guessed)

    def is_over(self) -> bool:
        return (self.assassin in self.guessed) or (len(self.remaining_blue) == 0)

    def outcome(self) -> str:
        if self.assassin in self.guessed:
            return "LOSS (assassin guessed)"
        if len(self.remaining_blue) == 0:
            return "WIN (all blue found)"
        return "IN_PROGRESS"

cfg = GameConfig()


## 4) Load fixed boards from `difficulty_boards.txt`

In [5]:
BOARD_HEADER_RE = re.compile(r"^BOARD\s+(\d+\.\d+)\s*$")

def _parse_list_line(line: str) -> List[str]:
    _, rhs = line.split(":", 1)
    return [w.strip().lower() for w in rhs.split(",") if w.strip()]

def load_difficulty_boards(path: str) -> Dict[str, Dict]:
    boards: Dict[str, Dict] = {}
    cur_id = None
    cur = None

    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue

            m = BOARD_HEADER_RE.match(line)
            if m:
                if cur_id is not None:
                    boards[cur_id] = cur
                cur_id = m.group(1)
                cur = {"our": [], "opp": [], "assassin": None, "neutral": []}
                continue

            if cur is None:
                continue

            if line.startswith("OUR:"):
                cur["our"] = _parse_list_line(line)
            elif line.startswith("OPP:"):
                cur["opp"] = _parse_list_line(line)
            elif line.startswith("ASSASSIN:"):
                cur["assassin"] = line.split(":", 1)[1].strip().lower()
            elif line.startswith("NEUTRAL:"):
                cur["neutral"] = _parse_list_line(line)

    if cur_id is not None and cur is not None:
        boards[cur_id] = cur

    # validation: 25 unique words / board
    for bid, b in boards.items():
        assert len(b["our"]) == 8, (bid, "OUR must have 8 words")
        assert len(b["opp"]) == 9, (bid, "OPP must have 9 words")
        assert b["assassin"] is not None, (bid, "ASSASSIN missing")
        assert len(b["neutral"]) == 7, (bid, "NEUTRAL must have 7 words")

        all_words = set(b["our"]) | set(b["opp"]) | set(b["neutral"]) | {b["assassin"]}
        assert len(all_words) == 25, (bid, "board must have 25 unique words")

    return boards

BOARDS_PATH = "/Users/nadavissacof/Documents/DSC253/codenames/difficulty_boards.txt"  # provided in this environment
boards = load_difficulty_boards(BOARDS_PATH)
print("Loaded boards:", len(boards))
print("Example board IDs:", list(boards.keys())[:10])

def level_of(board_id: str) -> int:
    return int(board_id.split(".")[0])

def make_game_from_board(board: Dict) -> GameState:
    blue = set(board["our"])
    assassin = board["assassin"]
    # Treat OPP + NEUTRAL as the same neutral category
    bystanders = set(board["opp"]) | set(board["neutral"])
    return GameState(blue=blue, assassin=assassin, bystanders=bystanders, guessed=set())


Loaded boards: 50
Example board IDs: ['1.1', '1.2', '1.3', '1.4', '1.5', '2.1', '2.2', '2.3', '2.4', '2.5']


## 5) Spymaster + Operative policies (Word2Vec)

In [6]:
def choose_clue_number_margin(
    model: KeyedVectors,
    clue: str,
    remaining_blue: Set[str],
    danger_words: Set[str],
    cap: int,
    margin: float = 0.07,
    min_sim: float = 0.10
) -> int:
    blue_sims = []
    for w in remaining_blue:
        if in_vocab(model, clue) and in_vocab(model, w):
            blue_sims.append(cosine_sim(model, clue, w))
    blue_sims.sort(reverse=True)

    danger_sims = []
    for w in danger_words:
        if in_vocab(model, clue) and in_vocab(model, w):
            danger_sims.append(cosine_sim(model, clue, w))
    best_danger = max(danger_sims) if danger_sims else -1.0

    k_best = 1
    for k in range(1, min(cap, len(blue_sims)) + 1):
        kth_blue = blue_sims[k-1]
        if kth_blue >= min_sim and (kth_blue - best_danger) >= margin:
            k_best = k
        else:
            break

    return max(1, k_best)

def clue_score_margin(
    model: KeyedVectors,
    clue: str,
    remaining_blue: Set[str],
    danger_words: Set[str],
    topk_blue: int = 3,
) -> float:
    blue_sims = [cosine_sim(model, clue, w) for w in remaining_blue if in_vocab(model, w)]
    if not blue_sims:
        return -1e9
    blue_sims.sort(reverse=True)
    reward = float(np.mean(blue_sims[: min(topk_blue, len(blue_sims))]))

    danger_sims = [cosine_sim(model, clue, w) for w in danger_words if in_vocab(model, w)]
    penalty = max(danger_sims) if danger_sims else -1.0

    return reward - penalty

def spymaster_clue(
    model: KeyedVectors,
    state: GameState,
    cfg: GameConfig,
    alpha: float = 2.0,
    beta: float = 1.0,
    pick_topk: int = 5
) -> Tuple[str, int, Dict]:
    B = sorted(state.remaining_blue)
    N = sorted(state.bystanders)
    A = state.assassin

    vB = mean_vector(model, B)
    vN = mean_vector(model, N)
    vA = model[A] if in_vocab(model, A) else np.zeros(model.vector_size, dtype=np.float32)

    target = vB - alpha * vA - beta * vN
    raw_candidates = [w for (w, _) in model.similar_by_vector(target, topn=cfg.clue_candidate_topn)]

    legal = [w for w in raw_candidates if is_legal_clue(w, state.board_all, model, cfg.illegal_clue_similarity_threshold)]
    if not legal:
        raw_candidates = [w for (w, _) in model.similar_by_vector(vB, topn=cfg.clue_candidate_topn)]
        legal = [w for w in raw_candidates if is_legal_clue(w, state.board_all, model, cfg.illegal_clue_similarity_threshold)]

    if not legal:
        # last resort: any legal vocab word
        for w in model.key_to_index.keys():
            if is_legal_clue(w, state.board_all, model, cfg.illegal_clue_similarity_threshold):
                legal = [w]
                raw_candidates = [w]
                break

    danger = set(state.bystanders) | {state.assassin}

    unused = [w for w in legal if w not in state.used_clues]
    pool = unused if unused else legal

    scored = [(w, clue_score_margin(model, w, state.remaining_blue, danger)) for w in pool]
    scored.sort(key=lambda x: x[1], reverse=True)

    k = min(pick_topk, len(scored))
    if k == 0:
        clue = legal[0]
        clue_score = None
    else:
        idx = int(np.random.randint(0, k))
        clue, clue_score = scored[idx]

    state.used_clues.add(clue)

    number = choose_clue_number_margin(model, clue, state.remaining_blue, danger, cfg.clue_number_cap)

    debug = {
        "remaining_blue": B,
        "assassin": A,
        "top_candidates": raw_candidates[:10],
        "legal_candidates": legal[:10],
        "scored_top10": scored[:10],
        "chosen_clue": clue,
        "chosen_clue_score": clue_score,
        "chosen_number": number,
    }
    return clue, number, debug

def operative_rank_guesses(model: KeyedVectors, clue: str, state: GameState) -> List[Tuple[str, float]]:
    ranked = []
    for w in sorted(state.remaining_unguessed):
        if in_vocab(model, clue) and in_vocab(model, w):
            ranked.append((w, cosine_sim(model, clue, w)))
        else:
            ranked.append((w, -1.0))
    ranked.sort(key=lambda x: x[1], reverse=True)
    return ranked

def operative_take_turn(model: KeyedVectors, clue: str, number: int, state: GameState, cfg: GameConfig) -> Dict:
    ranked = operative_rank_guesses(model, clue, state)
    max_guesses = number + (1 if cfg.operative_extra_guess else 0)
    guesses = [w for (w, _) in ranked[:max_guesses]]

    turn_log = {"clue": clue, "number": number, "guesses": [], "events": []}

    for g in guesses:
        state.guessed.add(g)
        if g == state.assassin:
            turn_log["guesses"].append((g, "ASSASSIN"))
            turn_log["events"].append("Hit assassin. Game over.")
            break
        elif g in state.blue:
            turn_log["guesses"].append((g, "BLUE"))
        else:
            turn_log["guesses"].append((g, "NEUTRAL"))
            turn_log["events"].append("Missed (neutral). Ending turn.")
            break

    return turn_log


## 6) Run a game on one board (optional demo)

In [7]:
def play_game_on_state(model: KeyedVectors, cfg: GameConfig, state: GameState, max_turns: int = 30, verbose: bool = True):
    turn = 0
    history = []

    if verbose:
        print("=== BOARD (operative sees) ===")
        print(sorted(state.board_all))
        print("\n(secret) BLUE:", sorted(state.blue))
        print("(secret) ASSASSIN:", state.assassin)

    while not state.is_over() and turn < max_turns:
        turn += 1
        clue, num, dbg = spymaster_clue(model, state, cfg)
        log = operative_take_turn(model, clue, num, state, cfg)
        history.append(log)

        if verbose:
            print(f"\n--- Turn {turn} ---")
            print(f"Spymaster clue: {clue} {num}")
            print("Operative guesses:", ", ".join([f"{w}({tag})" for w, tag in log["guesses"]]))
            if log["events"]:
                print("Notes:", " | ".join(log["events"]))
            print(f"Remaining blue: {len(state.remaining_blue)} -> {sorted(state.remaining_blue)}")

    if verbose:
        print("\n=== RESULT ===")
        print(state.outcome())
        print("Turns:", turn)

    return state, history

# Demo: pick a board id and run
board_id = list(boards.keys())[0]
state = make_game_from_board(boards[board_id])
_ = play_game_on_state(w2v, cfg, state, max_turns=30, verbose=True)


=== BOARD (operative sees) ===
['ball', 'battery', 'bear', 'bell', 'card', 'cat', 'chair', 'dog', 'eagle', 'engine', 'hawk', 'laser', 'lion', 'microscope', 'note', 'paper', 'poison', 'robot', 'satellite', 'screen', 'server', 'shark', 'table', 'telescope', 'whale']

(secret) BLUE: ['bear', 'cat', 'dog', 'eagle', 'hawk', 'lion', 'shark', 'whale']
(secret) ASSASSIN: poison

--- Turn 1 ---
Spymaster clue: otters 7
Operative guesses: whale(BLUE), shark(BLUE), cat(BLUE), lion(BLUE), dog(BLUE), eagle(BLUE), bear(BLUE)
Remaining blue: 1 -> ['hawk']

--- Turn 2 ---
Spymaster clue: maverick 1
Operative guesses: hawk(BLUE)
Remaining blue: 0 -> []

=== RESULT ===
WIN (all blue found)
Turns: 2


## 7) Evaluate all boards and summarize per board + per level

In [23]:
def evaluate_all_boards(model: KeyedVectors, cfg: GameConfig, boards: Dict[str, Dict], max_turns: int = 30):
    rows = []

    for bid, b in boards.items():
        lvl = level_of(bid)
        st = make_game_from_board(b)
        end_state, hist = play_game_on_state(model, cfg, st, max_turns=max_turns, verbose=False)

        turns = max(1, len(hist))
        blue_found = len(end_state.blue & end_state.guessed)
        accuracy = blue_found / len(end_state.blue)  # /8
        words_per_turn = blue_found / turns
        assassin_hit = int(end_state.assassin in end_state.guessed)
        win = int((blue_found == len(end_state.blue)) and (assassin_hit == 0))

        rows.append({
            "board_id": bid,
            "level": lvl,
            "turns": turns,
            "blue_found": blue_found,
            "accuracy": accuracy,
            "words_per_turn": words_per_turn,
            "assassin_hit": assassin_hit,
            "outcome": end_state.outcome(),
            "win": win,
        })

    per_board = pd.DataFrame(rows).sort_values(["level", "board_id"]).reset_index(drop=True)

    per_level = (
        per_board.groupby("level")
        .agg(
            n_boards=("board_id", "count"),
            avg_turns=("turns", "mean"),
            avg_words_per_turn=("words_per_turn", "mean"),
            avg_accuracy=("accuracy", "mean"),
            assassin_rate=("assassin_hit", "mean"),
            win_rate=("win", "mean"),
        )
        .reset_index()
        .sort_values("level")
    )

    return per_board, per_level

per_board_df, per_level_df = evaluate_all_boards(w2v, cfg, boards, max_turns=30)

# --- Add Overall row to per_level_df ---

overall_row = {
    "level": "Overall",
    "n_boards": per_board_df["board_id"].count(),
    "avg_turns": per_board_df["turns"].mean(),
    "avg_words_per_turn": (
        per_board_df["blue_found"].sum() / per_board_df["turns"].sum()
    ),
    "avg_accuracy": (
        per_board_df["blue_found"].sum() / (8 * per_board_df["board_id"].count())
    ),
    "assassin_rate": per_board_df["assassin_hit"].mean(),
    "win_rate": per_board_df["win"].mean(),
}

per_level_with_overall = pd.concat(
    [per_level_df, pd.DataFrame([overall_row])],
    ignore_index=True
)

per_level_with_overall

# print("Per-level summary:")
# display(
#     per_level_df.style.format({
#         "avg_turns": "{:.2f}",
#         "avg_words_per_turn": "{:.3f}",
#         "avg_accuracy": "{:.3f}",
#         "assassin_rate": "{:.3f}",
#         "win_rate": "{:.3f}",
#     })
# )

# print("\nPer-board results:")
# display(
#     per_board_df.head(50).style.format({
#         "accuracy": "{:.3f}",
#         "words_per_turn": "{:.3f}",
#     })
# )



,level,n_boards,avg_turns,avg_words_per_turn,avg_accuracy,assassin_rate,win_rate
0,1,5,2.60,4.053333,1.0,0.0,1.0
1,2,5,3.40,2.400000,1.0,0.0,1.0
2,3,5,3.80,2.186667,1.0,0.0,1.0
3,4,5,4.00,2.106667,1.0,0.0,1.0
4,5,5,6.00,1.363810,1.0,0.0,1.0
5,6,5,3.00,2.800000,1.0,0.0,1.0
6,7,5,4.40,1.840000,1.0,0.0,1.0
7,8,5,3.40,2.586667,1.0,0.0,1.0
8,9,5,3.40,3.253333,1.0,0.0,1.0
9,10,5,4.40,1.840000,1.0,0.0,1.0


### Notes on metrics

- **Turns**: number of spymaster clues given (capped at `max_turns`)
- **Accuracy**: `blue_found / 8`
- **Words/turn**: `blue_found / turns` (blue words only)
- **Assassin rate**: fraction of boards where assassin was guessed


In [9]:
def assassin_rank_report(model: KeyedVectors, clue: str, state: GameState):
    ranked = operative_rank_guesses(model, clue, state)  # list of (word, sim) for remaining unguessed
    words = [w for (w, _) in ranked]
    sims  = {w:s for (w,s) in ranked}

    a = state.assassin
    if a in words:
        r = words.index(a) + 1
        a_sim = sims[a]
    else:
        r = None
        a_sim = None

    top1_w, top1_s = ranked[0]
    top2_w, top2_s = ranked[1] if len(ranked) > 1 else (None, None)

    return {
        "assassin_rank": r,
        "assassin_sim": a_sim,
        "top1": (top1_w, top1_s),
        "top2": (top2_w, top2_s),
        "top_gap": (top1_s - top2_s) if top2_s is not None else None,
        "top10": ranked[:10],
    }

def play_game_with_debug(model: KeyedVectors, cfg: GameConfig, board_id: str, boards: Dict[str, Dict], max_turns: int = 50):
    state = make_game_from_board(boards[board_id])
    history = []
    print("BOARD:", board_id, "LEVEL:", level_of(board_id))
    print("Assassin:", state.assassin)
    print("Blue:", sorted(state.blue))
    print()

    turn = 0
    while not state.is_over() and turn < max_turns:
        turn += 1
        clue, num, dbg = spymaster_clue(model, state, cfg)

        # BEFORE guessing, measure assassin rank for this clue
        rep = assassin_rank_report(model, clue, state)

        log = operative_take_turn(model, clue, num, state, cfg)
        history.append((clue, num, rep, log))

        print(f"Turn {turn:02d}: clue={clue} {num} | assassin_rank={rep['assassin_rank']} assassin_sim={rep['assassin_sim']:.3f}" 
              if rep["assassin_rank"] is not None else
              f"Turn {turn:02d}: clue={clue} {num} | assassin already guessed?")

        print("   top1:", rep["top1"], "top2:", rep["top2"], "gap:", None if rep["top_gap"] is None else round(rep["top_gap"], 3))
        print("   guesses:", ", ".join([f"{w}({tag})" for w, tag in log["guesses"]]))
        if log["events"]:
            print("   events:", " | ".join(log["events"]))
        print("   remaining_blue:", len(state.remaining_blue))
        print()

        if state.assassin in state.guessed:
            print(">>> Assassin was guessed on this turn.")
            break

    print("Outcome:", state.outcome(), "| Turns:", turn, "| Blue found:", len(state.blue & state.guessed), "/ 8")
    return state, history

In [21]:
_ = play_game_with_debug(w2v, cfg, board_id="9.3", boards=boards, max_turns=60)

BOARD: 9.3 LEVEL: 9
Assassin: water
Blue: ['fish', 'pool', 'seal', 'shark', 'spring', 'stream', 'wave', 'whale']

Turn 01: clue=whaling 3 | assassin_rank=16 assassin_sim=0.041
   top1: ('whale', 0.540771484375) top2: ('shark', 0.3415672779083252) gap: 0.199
   guesses: whale(BLUE), shark(BLUE), fish(BLUE)
   remaining_blue: 5

Turn 02: clue=groundswell 2 | assassin_rank=10 assassin_sim=0.034
   top1: ('wave', 0.4383537769317627) top2: ('stream', 0.1572263538837433) gap: 0.281
   guesses: wave(BLUE), stream(BLUE)
   remaining_blue: 3

Turn 03: clue=monthlong 1 | assassin_rank=5 assassin_sim=0.037
   top1: ('spring', 0.42528241872787476) top2: ('seal', 0.09292470663785934) gap: 0.332
   guesses: spring(BLUE)
   remaining_blue: 2

Turn 04: clue=secures 1 | assassin_rank=16 assassin_sim=0.013
   top1: ('seal', 0.3592017889022827) top2: ('table', 0.10795539617538452) gap: 0.251
   guesses: seal(BLUE)
   remaining_blue: 1

Turn 05: clue=weeded 1 | assassin_rank=4 assassin_sim=0.055
   top1: 

In [11]:
board_id="3.5"
b = boards[board_id]
all_words = set(b["our"]) | set(b["opp"]) | set(b["neutral"]) | {b["assassin"]}
oov = sorted([w for w in all_words if w not in w2v.key_to_index])
len(oov), oov[:30]

(0, [])

In [22]:
def compute_overall_results(per_board_df):
    total_boards = len(per_board_df)
    total_turns = per_board_df["turns"].sum()
    total_blue_found = per_board_df["blue_found"].sum()
    total_possible_blue = total_boards * 8

    overall = {
        "n_boards": total_boards,
        "avg_turns": per_board_df["turns"].mean(),
        "avg_words_per_turn": (total_blue_found / total_turns),
        "avg_accuracy": (total_blue_found / total_possible_blue),
        "assassin_rate": per_board_df["assassin_hit"].mean(),
        "win_rate": per_board_df["win"].mean(),
    }

    return overall

overall_results = compute_overall_results(per_board_df)
overall_results

{'n_boards': 50,
 'avg_turns': 3.78,
 'avg_words_per_turn': 2.1164021164021163,
 'avg_accuracy': 1.0,
 'assassin_rate': 0.0,
 'win_rate': 1.0}